### Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

* Role - Identifies the message type (e.g. system, user)
* Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
* Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [2]:
# Setup the model
import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

Use text prompts when:

* You have a single, standalone request
* You don’t need conversation history
* You want minimal code complexity

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types

* System message - Tells the model how to behave and provide context for interactions
* Human message - Represents user input and interactions with the model
* AI message - Responses generated by the model, including text content, tool calls, and metadata
* Tool message - Represents the outputs of tool calls

#### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.

#### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

#### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

#### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [3]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are an Haiku poetry master."),
    HumanMessage("Write a poem about a kid's first day of school.")
]

response = model.invoke(messages)
response.content

'<think>\nOkay, the user wants a haiku about a kid\'s first day of school. Let me start by recalling the structure of a haiku—three lines with a 5-7-5 syllable count. \n\nFirst line: Maybe focus on the child\'s feelings. Words like "nervous" or "excited." Maybe mention the morning, like "Morning jitters creep," that\'s five syllables.\n\nSecond line: Needs seven syllables. Maybe something about the journey to school. "Backpack heavy with dreams," that\'s six. Hmm. Or "Backpack heavy with dreams and books," but that\'s eight. Wait, "Backpack heavy with dreams" is seven? Let me check. Back-pack-heavy-with-dreams. Yeah, seven syllables. \n\nThird line: The resolution or the action. Maybe arriving at school. "School doors swing wide." That\'s five syllables. \n\nLet me check the syllables again. Line one: 5, line two: 7, line three: 5. \n\nWait, "Morning jitters creep"—morning (2), jitters (2), creep (1). Total 5. Good. "Backpack heavy with dreams"—backpack (2), heavy (2), with (1), dreams

In [7]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

system_message = SystemMessage(
    """ You are a Senior Python developer with expertise in web frameworks.
    Always provide code example and explain your reasoning.
    Be concise but thorough in your answers.
    """
)

messages = [
    system_message,
    HumanMessage("How do I create a REST API?")
]

response = model.invoke(messages)
response.content

'<think>\nOkay, the user is asking how to create a REST API. Let me break this down step by step. First, I need to decide which framework to use. The user mentioned being a Python developer, so Flask or Django are common choices. Flask is lighter and more flexible for a simple API, so I\'ll go with that.\n\nNext, I should outline the structure of a basic REST API. The essential parts are endpoints for creating, reading, updating, and deleting resources. Let\'s pick a simple resource, like a To-Do item. Each item will have an ID, a title, and a description.\n\nI\'ll start by setting up a Flask app. Then, create routes for each CRUD operation. For example, a POST to /todo to create an item, GET to /todo/<id> to retrieve it, etc. I\'ll use a list to simulate a database for simplicity, which is good for demos but not for production.\n\nI need to handle HTTP methods correctly. For instance, POST for creating, GET for reading, PUT for updating, and DELETE for removing. Each route will have c

In [12]:
## Message Metadata
from langchain.messages import SystemMessage, HumanMessage, AIMessage

system_message = SystemMessage(
    """ You are a helpful assistant. Always start your replies politely by saying Hello <name of the human>. Also end the reply by mentioning the human provided id and random_metadata.
    """
)
human_message = HumanMessage(
    content="How are you?",
    name="Dinesh",
    id="human_msg_1",
    random_metadata="ABC",
)

messages = [
    system_message,
    human_message
]
print(messages)
response = model.invoke(messages)
response


[SystemMessage(content=' You are a helpful assistant. Always start your replies politely by saying Hello <name of the human>. Also end the reply by mentioning the human provided id and random_metadata.\n    ', additional_kwargs={}, response_metadata={}), HumanMessage(content='How are you?', additional_kwargs={}, response_metadata={}, name='Dinesh', id='human_msg_1', random_metadata='ABC')]


AIMessage(content='<think>\nOkay, the user asked, "How are you?" I need to respond politely as per the instructions. First, start with "Hello" followed by their name. Since the user didn\'t provide a name, maybe just use a placeholder like [Name]? Wait, the example in the instructions shows using the actual name. But the user here didn\'t mention their name. Hmm, maybe the system is designed to have the name provided by the user in the query. Let me check the initial message again. The user wrote "Hello" and then their message. Oh, right, the user might have included their name in the message, but in this case, they just asked "How are you?" without a name.\n\nWait, the instructions say to always start replies politely by saying Hello [name of the human]. So if the user didn\'t provide a name, maybe I should just say Hello and then proceed. Or perhaps the name is part of the system input. Wait, looking at the example, the user message here is just "How are you?" without a name. So mayb

In [14]:
from langchain.messages import AIMessage, ToolMessage

ai_msg = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "Denver"},
        "id": "call_123"
    }]
)

weather_result = "Sunny, 77 degrees Fahrenheit"

tool_msg = ToolMessage(
    content=weather_result,
    tool_call_id = "call_123"
)

messages = [
    HumanMessage("What is the weather in Denver?"),
    ai_msg,
    tool_msg
]
response = model.invoke(messages)
response.content

'<think>\nOkay, the user asked for the weather in Denver. I called the get_weather function with the location set to Denver. The response came back as sunny and 77°F. Now I need to present this information in a clear and friendly way. Let me make sure to mention the temperature and the weather condition. Maybe add a sentence about it being a nice day to encourage outdoor activities. Keep it concise but helpful. Let me check for any typos. Alright, that should cover it.\n</think>\n\nThe weather in Denver is currently sunny with a temperature of 77°F. It looks like a perfect day to enjoy outdoor activities! 🌞'